In [1]:
!pip install -q trl optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 15.0 MB/s eta 0:00:00


In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
import warnings
warnings.filterwarnings("ignore")
 
import os
import gc
import json
import pandas as pd
import torch
import optuna
from optuna.samplers import TPESampler
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer
from kaggle_secrets import UserSecretsClient

import wandb

In [4]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [5]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
    print(f"VRAM total: {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")

GPU: Tesla T4
VRAM free: 15.5 GB
VRAM total: 15.6 GB


In [6]:
MODEL_ID     = "mistralai/Mistral-7B-Instruct-v0.3"
N_TRIALS     = 5
STUDY_NAME   = "mistral-qlora-hpo"
RESULTS_FILE = "/kaggle/working/optuna_results.json"
 
secret_label = "HF_TOKEN"
hf_token = UserSecretsClient().get_secret(secret_label)
os.environ["HF_TOKEN"] = hf_token

In [7]:
train_df = pd.read_csv("/kaggle/input/datasets/linhtron123/legaldata/train_split.csv")
val_df   = pd.read_csv("/kaggle/input/datasets/linhtron123/legaldata/val_split.csv")
 
def format_example(row):
    manipulative      = int(row['Manipulative'])          if pd.notna(row['Manipulative'])          else 0
    primary_manipulator = str(row['Primary Manipulator']) if pd.notna(row['Primary Manipulator'])    else "None"
    techniques        = str(row['Manipulation Techniques']) if pd.notna(row['Manipulation Techniques']) else "None"
    dialogue          = str(row['Dialogue']).strip()
 
    instruction = (
        "Analyze the following dialogue and determine:\n"
        "1. Is there manipulation present? (0 = No, 1 = Yes)\n"
        "2. Who is the primary manipulator? (Plaintiff, Defendant, or None)\n"
        "3. What manipulation techniques are used? (or None if not manipulative)"
    )
    response = (
        f"Manipulation Present: {manipulative}\n"
        f"Primary Manipulator: {primary_manipulator}\n"
        f"Manipulation Techniques: {techniques}"
    )
    return {"text": f"[INST] {instruction}\n\nDialogue: {dialogue} [/INST] {response}"}
 
formatted_train = [format_example(r) for _, r in train_df.iterrows()]
formatted_val   = [format_example(r) for _, r in val_df.iterrows()]
 
train_dataset = Dataset.from_pandas(pd.DataFrame(formatted_train))
eval_dataset  = Dataset.from_pandas(pd.DataFrame(formatted_val))
 
print(f"Train: {len(train_dataset)} | Val: {len(eval_dataset)}")

Train: 720 | Val: 155


In [8]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right"

Loading tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
def objective(trial: optuna.Trial) -> float:
    learning_rate = trial.suggest_float("learning_rate",  1e-5, 5e-4, log=True)
    lora_r        = trial.suggest_categorical("lora_r",   [8, 16, 32])
    lora_alpha    = trial.suggest_categorical("lora_alpha", [16, 32, 64])
    lora_dropout  = trial.suggest_float("lora_dropout",   0.0,  0.2, step=0.05)
    warmup_ratio  = trial.suggest_float("warmup_ratio",   0.03, 0.15, step=0.01)
    per_device_bs = trial.suggest_categorical("per_device_train_batch_size", [1, 2])
    grad_accum    = trial.suggest_categorical("gradient_accumulation_steps", [4, 8])

    print(f"\n{'='*55}")
    print(f"TRIAL {trial.number:02d} | lr={learning_rate:.2e} | r={lora_r} | "
          f"alpha={lora_alpha} | dropout={lora_dropout} | "
          f"bs={per_device_bs} | accum={grad_accum} | warmup={warmup_ratio}")
    print(f"{'='*55}")

    gc.collect()
    try:
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    except Exception:
        pass

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
    )
    model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)

    training_args = TrainingArguments(
        output_dir=f"/kaggle/working/trial_{trial.number}",
        per_device_train_batch_size=per_device_bs,
        gradient_accumulation_steps=grad_accum,
        optim="paged_adamw_8bit",
        learning_rate=learning_rate,
        num_train_epochs=1,
        warmup_ratio=warmup_ratio,
        fp16=False,
        bf16=False,
        gradient_checkpointing=False,
        max_grad_norm=0.3,
        eval_strategy="steps",
        eval_steps=30,
        save_strategy="steps",
        save_steps=30,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1,
        report_to="none",
        push_to_hub=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    eval_loss = float("inf")
    try:                                      
        trainer.train()
        eval_results = trainer.evaluate()
        eval_loss = eval_results["eval_loss"]
        print(f"Trial {trial.number:02d} eval_loss = {eval_loss:.4f}")
    except Exception as e:
        print(f"Trial {trial.number:02d} FAILED: {e}")
        eval_loss = float("inf")
    finally:
        del trainer
        del model
        gc.collect()
        try:
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            if hasattr(torch.cuda, 'reset_peak_memory_stats'):
                torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass
        gc.collect()
        free = torch.cuda.mem_get_info()[0] / 1e9
        print(f"  VRAM free sau trial: {free:.1f} GB")

    return eval_loss                              

In [ ]:
print(f"\nBắt đầu Optuna HPO: {N_TRIALS} trials")
print("Thuật toán: TPE (Tree-structured Parzen Estimator)\n")
 
sampler = TPESampler(seed=42)
study = optuna.create_study(
    direction="minimize",
    study_name=STUDY_NAME,
    sampler=sampler,
)
 
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
    catch=(Exception, RuntimeError, torch.cuda.OutOfMemoryError), 
)

[I 2026-05-14 03:06:19,449] A new study created in memory with name: mistral-qlora-hpo



Bắt đầu Optuna HPO: 5 trials
Thuật toán: TPE (Tree-structured Parzen Estimator)



  0%|          | 0/5 [00:00<?, ?it/s]


TRIAL 00 | lr=4.33e-05 | r=8 | alpha=16 | dropout=0.2 | bs=1 | accum=4 | warmup=0.1


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Adding EOS to eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
30,1.691483,1.698536
60,1.654924,1.702568
90,1.626667,1.664042


Trial 04 eval_loss = 1.6640
  VRAM free sau trial: 12.1 GB
[I 2026-05-14 10:51:01,512] Trial 4 finished with value: 1.6640431880950928 and parameters: {'learning_rate': 0.0003946212980759095, 'lora_r': 32, 'lora_alpha': 32, 'lora_dropout': 0.05, 'warmup_ratio': 0.08, 'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 4}. Best is trial 3 with value: 1.6468487977981567.


In [11]:
best = study.best_trial
print("\n" + "="*55)
print("KẾT QUẢ TỐT NHẤT")
print("="*55)
print(f"Eval Loss: {best.value:.4f}")
print("Hyperparameters:")
for k, v in best.params.items():
    print(f"  {k}: {v}")
 
# Lưu toàn bộ kết quả
results = {
    "best_eval_loss": best.value,
    "best_params": best.params,
    "all_trials": [
        {
            "trial": t.number,
            "eval_loss": t.value,
            "params": t.params,
            "state": str(t.state),
        }
        for t in study.trials
    ],
}
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nKết quả lưu tại: {RESULTS_FILE}")


KẾT QUẢ TỐT NHẤT
Eval Loss: 1.6468
Hyperparameters:
  learning_rate: 5.595074635794791e-05
  lora_r: 16
  lora_alpha: 16
  lora_dropout: 0.05
  warmup_ratio: 0.09
  per_device_train_batch_size: 1
  gradient_accumulation_steps: 4

Kết quả lưu tại: /kaggle/working/optuna_results.json


In [12]:
secret_label_wandb = "WANDB_API_KEY"
wandb_key = UserSecretsClient().get_secret(secret_label_wandb)
wandb.login(key=wandb_key)
os.environ["WANDB_PROJECT"] = "mistral-finetune"
 
print("\n" + "="*55)
print("TRAINING CUỐI VỚI BEST PARAMS (2 EPOCHS)")
print("="*55)
 
p = best.params
torch.cuda.empty_cache()
gc.collect()
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
 
final_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", token=hf_token
)
final_model = prepare_model_for_kbit_training(final_model)
 
final_lora_cfg = LoraConfig(
    r=p["lora_r"],
    lora_alpha=p["lora_alpha"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=p["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
)
final_model = get_peft_model(final_model, final_lora_cfg)
final_model.print_trainable_parameters()
 
final_training_args = TrainingArguments(
    output_dir="./results_best",
    per_device_train_batch_size=p["per_device_train_batch_size"],
    gradient_accumulation_steps=p["gradient_accumulation_steps"],
    optim="paged_adamw_8bit",
    learning_rate=p["learning_rate"],
    num_train_epochs=2,
    warmup_ratio=p["warmup_ratio"],
    fp16=False,   # ← sửa lại
    bf16=False,   # ← thêm
    gradient_checkpointing=False,
    max_grad_norm=0.3,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
    report_to="wandb",
    push_to_hub=False,
)
 
final_trainer = SFTTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)
 
try:
    result = final_trainer.train()
    print(f"\nFinal training loss: {result.training_loss:.4f}")
    final_trainer.save_model("./final_model_best")
    tokenizer.save_pretrained("./final_model_best")
    print("Model lưu tại ./final_model_best")
except Exception as e:
    print(f"Final training thất bại: {e}")
    raise e
finally:
    wandb.finish()
    print("W&B run finished")
    try:
        final_trainer.save_model()
    except Exception:
        pass
    del final_model, final_trainer
    gc.collect()
    try:
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, 'reset_peak_memory_stats'):
            torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass
    gc.collect()
    free = torch.cuda.mem_get_info()[0] / 1e9
    print(f"VRAM free sau training: {free:.1f} GB")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: luongthilinh2702 (ltl2702) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



TRAINING CUỐI VỚI BEST PARAMS (2 EPOCHS)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


...


Final training loss: 1.6165


wandb: updating run metadata


Model lưu tại ./final_model_best


wandb: uploading wandb-summary.json; uploading config.yaml; uploading output.log
wandb: 
wandb: Run history:
wandb:             eval/entropy ▇██▁▃▇▁
wandb:                eval/loss █▄▃▃▁▁▁
wandb: eval/mean_token_accuracy ▁▅▇▆█▇█
wandb:          eval/num_tokens ▁▂▃▅▆▇█
wandb:             eval/runtime ▇█▆▆▅▁▂
wandb:  eval/samples_per_second ▁▁▁▁▁██
wandb:    eval/steps_per_second ▁▁▁▁▁▁▁
wandb:            train/entropy ▁▇█▇▅▄▅▅▆▃▆▄▄▇▇▃▃▄▆▂▄▁▂▃▃▃▅▃▂▄▃▃▃▃▂▃
wandb:              train/epoch ▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
wandb:        train/global_step ▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
wandb:                       +5 ...
wandb: 
wandb: Run summary:
wandb:             eval/entropy 1.57272
wandb:                eval/loss 1.64142
wandb: eval/mean_token_accuracy 0.61995
wandb:          eval/num_tokens 1231212.0
wandb:             eval/runtime 255.0737
wandb:  eval/samples_per_second 0.608
wandb:    eval/steps_per_second 0.078
wandb:               total_flos 5.4387269240832e+16
w

W&B run finished
VRAM free sau training: 11.6 GB
